### Vector db

ChromaDB

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
client=chromadb.Client()

In [ ]:
client.delete_collection("my_collection")

In [3]:
collection=client.create_collection("my_collection")

documents = [
    "Machine learning basics",
    "Deep learning with transformers",
    "Python programming tutorial"
]
ids = ["1", "2", "3"]

embeddings = model.encode(documents).tolist()

In [20]:
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings
)

print("Documents added!")


: 

In [4]:
query = "neural networks"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)

In [5]:
print("\nTop Matches:\n")

for doc in results['documents'][0]:
    print(doc)


Top Matches:



Persistent client chromaDB

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(
    path="./chroma_storage"
)

In [ ]:
collection = client.get_or_create_collection(
    name="my_collection"
)

documents = [
    "Machine learning basics",
    "Deep learning with transformers",
    "Python programming tutorial"
]

ids = ["1", "2", "3"]

embeddings = model.encode(documents).tolist()

In [ ]:
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings
)
print("Documents stored permanently!")

In [ ]:
query = "neural networks"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)

In [ ]:
print("\nTop Matches:\n")

for doc in results['documents'][0]:
    print(doc)

Pinecone

In [5]:
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import time
import os
key=os.getenv('PINECONE_API_KEY')
pc = Pinecone(api_key=key)
model = SentenceTransformer('all-MiniLM-L6-v2')

In [10]:
pc.delete_index("docs-index")

In [11]:
index_name = "docs-index"
if index_name not in pc.list_indexes().names():
    
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    print("Creating index...")
    time.sleep(10)
index = pc.Index(index_name)

Creating index...


In [12]:
documents = [
    "Machine learning basics",
    "Deep learning with transformers",
    "Python programming tutorial",
    "Vector databases for AI"
]

ids = ["1", "2", "3", "4"]

embeddings = model.encode(documents).tolist()

In [13]:
vectors = []

for i in range(len(documents)):
    vectors.append(
        (
            ids[i],
            embeddings[i],
            {"text": documents[i]}
        )
    )

index.upsert(vectors=vectors)

print("Vectors uploaded successfully!")

Vectors uploaded successfully!


In [14]:
query = "neural networks"

query_embedding = model.encode(query).tolist()

results = index.query(
    vector=query_embedding,
    top_k=2,
    include_metadata=True
)

In [15]:
print("\nTop Matches:\n")

for match in results['matches']:
    print(match['metadata']['text'])


Top Matches:

Machine learning basics
Vector databases for AI


### Metadata Filtering

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [10]:
# DOCUMENT DATASET
documents = [
    {
        "text": "Python generators improve memory efficiency in large data pipelines.",
        "metadata": {
            "source": "Official Python Docs",
            "date": "2025-01-10",
            "category": "python",
            "author": "Guido"
        }
    },

    {
        "text": "Python async programming enables concurrent API requests.",
        "metadata": {
            "source": "Python Blog",
            "date": "2025-02-15",
            "category": "python",
            "author": "Alice"
        }
    },

    {
        "text": "Java streams provide functional-style operations for collections.",
        "metadata": {
            "source": "Oracle Docs",
            "date": "2025-03-01",
            "category": "java",
            "author": "James"
        }
    },

    {
        "text": "Java multithreading improves performance in CPU-intensive applications.",
        "metadata": {
            "source": "Java Blog",
            "date": "2024-11-05",
            "category": "java",
            "author": "Bob"
        }
    },

    {
        "text": "Python decorators modify function behavior dynamically.",
        "metadata": {
            "source": "Advanced Python Guide",
            "date": "2025-04-12",
            "category": "python",
            "author": "Alice"
        }
    },

    {
        "text": "Machine learning models require careful feature engineering.",
        "metadata": {
            "source": "ML Research Journal",
            "date": "2025-02-20",
            "category": "machine_learning",
            "author": "Andrew"
        }
    },

    {
        "text": "Neural networks are trained using gradient descent optimization.",
        "metadata": {
            "source": "Deep Learning Book",
            "date": "2024-12-11",
            "category": "machine_learning",
            "author": "Ian"
        }
    }
]

In [11]:
# LOAD EMBEDDING MODEL
model = SentenceTransformer("all-MiniLM-L6-v2")

# CREATE EMBEDDINGS
texts = [doc["text"] for doc in documents]
doc_embeddings = model.encode(texts)
print("\nEmbeddings created.")
print(f"Total documents: {len(documents)}")


Embeddings created.
Total documents: 7


In [12]:
# BASIC VECTOR SEARCH (NO FILTERING)
def similarity_search(query, top_k=3):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(
        query_embedding,
        doc_embeddings
    )[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "score": float(similarities[idx]),
            "text": documents[idx]["text"],
            "metadata": documents[idx]["metadata"]
        })
    return results

In [13]:
# VECTOR SEARCH WITH METADATA FILTERING
def filtered_similarity_search(query, metadata_filter, top_k=3):
    filtered_docs = []
    filtered_embeddings = []
    # Apply metadata filters
    for i, doc in enumerate(documents):
        matched = True
        for key, value in metadata_filter.items():
            if doc["metadata"].get(key) != value:
                matched = False
                break
        if matched:
            filtered_docs.append(doc)
            filtered_embeddings.append(doc_embeddings[i])
    if len(filtered_docs) == 0:
        return []
    filtered_embeddings = np.array(filtered_embeddings)
    query_embedding = model.encode([query])
    similarities = cosine_similarity(
        query_embedding,
        filtered_embeddings
    )[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "score": float(similarities[idx]),
            "text": filtered_docs[idx]["text"],
            "metadata": filtered_docs[idx]["metadata"]
        })
    return results


In [15]:
query = "How does Python improve concurrency?"
print("\n" + "=" * 60)
print("QUERY")
print("=" * 60)
print(query)


QUERY
How does Python improve concurrency?


In [16]:
# RETRIEVAL WITHOUT FILTERING
print("\n" + "=" * 70)
print("RETRIEVAL WITHOUT FILTERING")
print("=" * 70)
results_no_filter = similarity_search(query)
for i, result in enumerate(results_no_filter, start=1):
    print(f"\nRank #{i}")
    print(f"Similarity Score : {result['score']:.4f}")
    print(f"Source           : {result['metadata']['source']}")
    print(f"Date             : {result['metadata']['date']}")
    print(f"Category         : {result['metadata']['category']}")
    print(f"Author           : {result['metadata']['author']}")
    print(f"Document         : {result['text']}")


RETRIEVAL WITHOUT FILTERING

Rank #1
Similarity Score : 0.6997
Source           : Python Blog
Date             : 2025-02-15
Category         : python
Author           : Alice
Document         : Python async programming enables concurrent API requests.

Rank #2
Similarity Score : 0.5429
Source           : Official Python Docs
Date             : 2025-01-10
Category         : python
Author           : Guido
Document         : Python generators improve memory efficiency in large data pipelines.

Rank #3
Similarity Score : 0.4513
Source           : Java Blog
Date             : 2024-11-05
Category         : java
Author           : Bob
Document         : Java multithreading improves performance in CPU-intensive applications.


In [17]:
# RETRIEVAL WITH FILTERING
print("\n" + "=" * 70)
print("RETRIEVAL WITH FILTERING")
print("=" * 70)
# Restrict search only to Python documents
metadata_filter = {
    "category": "python"
}
results_filtered = filtered_similarity_search(
    query,
    metadata_filter
)
for i, result in enumerate(results_filtered, start=1):
    print(f"\nRank #{i}")
    print(f"Similarity Score : {result['score']:.4f}")
    print(f"Source           : {result['metadata']['source']}")
    print(f"Date             : {result['metadata']['date']}")
    print(f"Category         : {result['metadata']['category']}")
    print(f"Author           : {result['metadata']['author']}")
    print(f"Document         : {result['text']}")


RETRIEVAL WITH FILTERING

Rank #1
Similarity Score : 0.6997
Source           : Python Blog
Date             : 2025-02-15
Category         : python
Author           : Alice
Document         : Python async programming enables concurrent API requests.

Rank #2
Similarity Score : 0.5429
Source           : Official Python Docs
Date             : 2025-01-10
Category         : python
Author           : Guido
Document         : Python generators improve memory efficiency in large data pipelines.

Rank #3
Similarity Score : 0.3091
Source           : Advanced Python Guide
Date             : 2025-04-12
Category         : python
Author           : Alice
Document         : Python decorators modify function behavior dynamically.


In [18]:
# EVALUATION
def precision_at_k(results, expected_category):
    correct = 0
    for result in results:
        if result["metadata"]["category"] == expected_category:
            correct += 1
    return correct / len(results)
precision_without_filter = precision_at_k(
    results_no_filter,
    "python"
)
precision_with_filter = precision_at_k(
    results_filtered,
    "python"
)
print("\n" + "=" * 70)
print("PRECISION EVALUATION")
print("=" * 70)
print(f"\nPrecision WITHOUT filtering : "
      f"{precision_without_filter:.2f}")
print(f"Precision WITH filtering    : "
      f"{precision_with_filter:.2f}")
improvement = (
    precision_with_filter -
    precision_without_filter
)
print(f"\nPrecision Improvement       : "
      f"{improvement:.2f}")


PRECISION EVALUATION

Precision WITHOUT filtering : 0.67
Precision WITH filtering    : 1.00

Precision Improvement       : 0.33


### Recall, Precision and MRR

In [8]:
def recall_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    hits = len(set(retrieved_k) & set(relevant))
    return hits / len(relevant)

def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    hits = len(set(retrieved_k) & set(relevant))
    return hits / k

def reciprocal_rank(retrieved, relevant):
    for idx, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1 / idx
    return 0

# Example
retrieved = ["c8", "c1", "c2", "c9"]

relevant = ["c1", "c2"]

print("Recall@3:", recall_at_k(retrieved, relevant, 3))
print("Precision@3:", precision_at_k(retrieved, relevant, 3))
print("MRR:", reciprocal_rank(retrieved, relevant))

Recall@3: 1.0
Precision@3: 0.6666666666666666
MRR: 0.5


### Bi-Encoding and Cross-Encoding

In [ ]:
"""

---
title: Bi-Encoding
---
graph TD
    
    A[Document 1] --> D(Encoder 2)
    B[Query] --> E(Encoder 1)
    C[Document 2] --> F(Encoder 2)
    
    D --> H>Doc 1 Embedding]
    E --> G>Query Embedding]
    F --> I>Doc 2 Embedding]
    
    G & H --> J{{Cosine Similarity}}
    G & I --> K{{Cosine Similarity}}
    
    J --> L((Score 1))
    K --> M((Score 2))
    
    L & M --> N{Ranking \nTop-K}
    
"""


In [ ]:
"""

---
title: Cross-Encoding
---
graph TD
    A[Query] --> C(BERT-like Transformer)
    B[Candidate Doc] --> C
    C --> E>Classifier]
    E --> F((Relevance \nScore))

"""


In [ ]:
"""

---
title: Hybrid Encoding
---
graph TB
    A[Query] --> B(Bi-Encoder)
    C[(Vector DB)] --> B
    B --> D>Top-K Candidates]
    D --> E(Cross-Encoder)
    A --> E
    E --> F((Final Ranked \nResults))

"""